# Stubs y Drivers

**Objetivo:** Comprender los conceptos de **stub** y **driver**, sus diferencias, cuándo usarlos, y cómo implementarlos en Python para pruebas de integración.

---

## 1. ¿Qué son los dobles de prueba en integración?

En las pruebas unitarias usamos mocks para aislar cada módulo. En las pruebas de integración, a veces necesitamos reemplazar **temporalmente** algunos módulos mientras probamos otros de forma real. Los dos dobles fundamentales para integración son:

- **Stub:** Reemplaza un módulo **inferior** (dependencia que es llamada). Devuelve respuestas controladas.
- **Driver:** Reemplaza un módulo **superior** (el que llama). Invoca al módulo bajo prueba y verifica su comportamiento.

La diferencia clave es la **dirección de la dependencia**:

```
STUB:    Tests → [Módulo bajo prueba] → [STUB reemplaza dependencia inferior]
DRIVER:  [DRIVER reemplaza módulo superior] → [Módulo bajo prueba]
```

Además existen otros tipos de dobles importantes:

| Tipo | ¿Qué hace? | ¿Verifica interacciones? | Uso típico |
|------|-----------|--------------------------|------------|
| **Dummy** | Se pasa como parámetro pero nunca se usa | No | Rellenar firmas de métodos |
| **Stub** | Devuelve respuestas predefinidas | No | Controlar estado de dependencias |
| **Spy** | Registra las llamadas recibidas | Sí (a posteriori) | Verificar que se llamó con los argumentos correctos |
| **Mock** | Define expectativas y las verifica automáticamente | Sí (automáticamente) | Verificar protocolo de colaboración |
| **Fake** | Implementación simplificada pero funcional | No | Reemplazar infraestructura costosa (BD en memoria) |

---

## 2. Stub — Simulando módulos inferiores

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ──────────────────────────────────────────────────────
# STUB de Notifier
# Reemplaza el Notifier real (que tiene fallo aleatorio)
# con un doble controlado.
# ──────────────────────────────────────────────────────
class StubNotifier:
    """Stub que implementa la misma interfaz que Notifier."""

    def __init__(self, fail=False):
        self.messages = []
        self._fail = fail

    def send(self, message):
        if self._fail:
            raise ConnectionError("Error simulado: fallo de red")
        self.messages.append(message)  # registra para verificación


# ──────────────────────────────────────────────────────
# STUB de TaskStorage
# Reemplaza el storage real (que usa disco) con uno en memoria.
# ──────────────────────────────────────────────────────
class StubStorage:
    """Stub que implementa la misma interfaz que TaskStorage."""

    def __init__(self, initial_tasks=None, fail_on_save=False):
        self._tasks = initial_tasks if initial_tasks is not None else []
        self._fail_on_save = fail_on_save

    def load(self):
        return list(self._tasks)  # devuelve copia

    def save(self, tasks):
        if self._fail_on_save:
            raise IOError("Error simulado: no se pudo escribir en disco")
        self._tasks = list(tasks)


print("Stubs definidos correctamente.")
print("Características de un buen stub:")
print("  1. Implementa la misma interfaz que el módulo real.")
print("  2. Devuelve datos predecibles (sin aleatoriedad).")
print("  3. Puede configurarse para simular éxito o fallo.")

In [ ]:
from src.service import TaskService

# Escenario 1: Happy path
service = TaskService(StubStorage(), StubNotifier())
result = service.add_task("Estudiar stubs")
print(f"[Esc. 1] add_task retornó: {result}")

# Escenario 2: Tarea duplicada — el stub tiene la tarea preexistente
storage2 = StubStorage(initial_tasks=[{"title": "Estudiar stubs", "done": False}])
service2 = TaskService(storage2, StubNotifier())
result2 = service2.add_task("Estudiar stubs")
print(f"[Esc. 2] add_task (duplicado) retornó: {result2}")

# Escenario 3: Fallo del notifier — ¿qué pasa con la tarea?
storage3 = StubStorage()
service3 = TaskService(storage3, StubNotifier(fail=True))
try:
    service3.add_task("Tarea con notifier fallido")
except ConnectionError as e:
    print(f"[Esc. 3] Excepción: {e}")
    print(f"[Esc. 3] ¿Quedó guardada? {storage3._tasks}")
    print("         ⚠ Bug de integración: tarea guardada pero excepción propagada sin rollback.")

---

## 3. Driver — Orquestando módulos inferiores

Para probar un módulo de infraestructura (como `TaskStorage`) **sin** que `TaskService` esté implementado, escribimos un **driver**. El driver actúa como el módulo superior que aún no existe.

### ¿Cuándo usar un driver?
- Cuando queremos validar un módulo de infraestructura en profundidad antes de integrarlo.
- En el enfoque **Bottom-Up**, donde se prueban las capas bajas primero.

In [ ]:
import tempfile, json
from src.storage import TaskStorage

# ──────────────────────────────────────────────────────
# DRIVER para TaskStorage
# Prueba directamente el módulo de almacenamiento
# sin pasar por TaskService.
# ──────────────────────────────────────────────────────

def run_storage_driver():
    with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as f:
        filepath = f.name
    os.unlink(filepath)  # que TaskStorage lo cree limpio

    try:
        storage = TaskStorage(filepath)

        # Caso 1: Lista vacía inicial
        assert storage.load() == [], "Debe iniciar vacío"
        print("✓ Caso 1: storage inicia con lista vacía")

        # Caso 2: Guardar y recuperar
        storage.save([{"title": "Leer docs", "done": False}])
        tasks = storage.load()
        assert len(tasks) == 1 and tasks[0]["title"] == "Leer docs"
        print("✓ Caso 2: save/load preserva los datos")

        # Caso 3: save sobrescribe correctamente
        storage.save([{"title": "T1", "done": False}, {"title": "T2", "done": True}])
        tasks = storage.load()
        assert len(tasks) == 2 and tasks[1]["done"] is True
        print("✓ Caso 3: save sobrescribe la lista completa")

        # Caso 4: Persistencia entre instancias
        storage2 = TaskStorage(filepath)
        assert storage2.load() == tasks
        print("✓ Caso 4: datos persisten entre instancias diferentes")

        # Caso 5: El formato en disco es JSON legible
        with open(filepath) as f:
            disk_data = json.load(f)
        assert isinstance(disk_data, list)
        print("✓ Caso 5: formato en disco es JSON válido")

    finally:
        if os.path.exists(filepath):
            os.unlink(filepath)


run_storage_driver()
print("\nDriver completado: TaskStorage funciona correctamente de forma aislada.")

---

## 4. Spy — Verificando interacciones entre módulos

Un **spy** es como un stub que además *registra* las llamadas que recibe. Es especialmente útil para verificar que `TaskService` realmente interactuó con sus dependencias de la manera esperada.

In [ ]:
class SpyStorage:
    """Spy que registra todas las interacciones."""

    def __init__(self):
        self._tasks = []
        self.save_calls = []   # historial de argumentos pasados a save()
        self.load_count = 0

    def load(self):
        self.load_count += 1
        return list(self._tasks)

    def save(self, tasks):
        self.save_calls.append(list(tasks))
        self._tasks = list(tasks)


spy = SpyStorage()
service = TaskService(spy, StubNotifier())
service.add_task("Revisar PR")

print(f"load() fue llamado {spy.load_count} vez/veces")
print(f"save() fue llamado {len(spy.save_calls)} vez/veces")
print(f"save() recibió: {spy.save_calls[0]}")
print()

# Aserciones sobre el PROTOCOLO de colaboración
assert spy.load_count == 1, "service debe cargar antes de guardar"
assert len(spy.save_calls) == 1, "service debe llamar a save() exactamente una vez"
assert spy.save_calls[0] == [{"title": "Revisar PR", "done": False}]

print("El spy nos confirma CÓMO interactuó service con storage:")
print("  - El orden de las llamadas fue correcto (load → save).")
print("  - Los datos enviados a save() tienen el formato correcto.")

---

## 5. Cuándo usar cada doble

```
¿Necesito controlar la respuesta de la dependencia?
    └─ Sí → ¿La dependencia tiene lógica relevante que quiero ejecutar?
               ├─ Sí → FAKE (implementación simplificada pero funcional)
               └─ No → STUB (respuesta fija predefinida)
¿Necesito verificar que la dependencia fue llamada correctamente?
    ├─ Sí, verifico yo manualmente → SPY
    └─ Sí, quiero que falle automáticamente → MOCK
¿Solo necesito un objeto que 'ocupe el lugar' sin usarse?
    └─ DUMMY
```

| Enfoque de integración | ¿Usa Stubs? | ¿Usa Drivers? |
|------------------------|-------------|---------------|
| **Top-Down**           | Sí          | No            |
| **Bottom-Up**          | No          | Sí            |
| **Sandwich**           | Sí          | Sí            |

---

## 6. Ejercicio práctico

1. Crea un `StubStorage` que falle al llamar a `load()` lanzando `IOError`. Úsalo con `TaskService` para verificar cómo se comporta el servicio ante un error de lectura.

2. Implementa un `SpyNotifier` que registre todos los mensajes enviados. Úsalo para verificar que `add_task()` llama a `notifier.send()` con un mensaje que contiene el título de la tarea.

3. Guarda el stub en un archivo auxiliar y el driver en `tests/test_storage_driver.py`.